# 02 - Entrenar M_seg (segmentacion hoja vs fondo)

U-Net con encoder **ResNet50** (transfer learning ImageNet). **2 clases**: `0=fondo`, `1=hoja`.

La red solo decide **que es hoja y que es fondo**. La distincion sana/enferma y el tipo de patogeno los hacen M1/M2; la severidad por color se calcula en inferencia (estilo Edgar). Esto evita el error de marcar enferma por color (p.ej. plagas: hoja verde con huecos).

## Datos y splits
- **GT experto**: mascaras COCO en `splits/masks/_annotations.coco.json` (clase hoja). Split **75/25 (seed 42)**: 25% test held-out; del 75%, 15% val.
- **Volumen extra**: pseudo-mascaras de hoja (HSV) desde las imagenes de clasificacion (fondo simple) para mas datos de entrenamiento; el **test/val se evaluan solo contra COCO experto**.

## Metricas
- **Principal: recall de hoja** (robusto a anotacion parcial en canopy denso, donde hojas no anotadas cuentan como fondo).
- Dice/IoU vs experto (secundario; sub-estima si la anotacion es parcial).

## Normalizacion
- **Gray-World** (reproducible identica en Dart) antes de la red, igual que en inferencia.

In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib opencv-python-headless pycocotools

In [ ]:
import json, time
import numpy as np
import cv2
import tensorflow as tf
import albumentations as A
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.utils import Sequence

tf.random.set_seed(42)
np.random.seed(42)
print('GPU:', tf.config.list_physical_devices('GPU'))

IMG_SIZE = (256, 256)
NUM_CLASSES = 2
BATCH = 8
EPOCHS_P1 = 20
EPOCHS_P2 = 40
LR_P1 = 1e-3
LR_P2 = 1e-4
L2_DEFAULT = 2e-4
TEST_FRACTION = 0.25
VAL_FRACTION = 0.15
PSEUDO_PER_EPOCH = 400

SPLIT = Path('./splits')
MASKS_DIR = SPLIT / 'masks'
COCO_ANN = MASKS_DIR / '_annotations.coco.json'
CLS_DIR = SPLIT / 'clasificacion_patogeno' / 'train'
OUT = Path('./outputs')
OUT.mkdir(exist_ok=True)

In [ ]:
_HSV_GREEN_LOW  = np.array([20, 25, 30])
_HSV_GREEN_HIGH = np.array([90, 255, 255])
_HSV_LESION_LOW  = np.array([0, 40, 25])
_HSV_LESION_HIGH = np.array([35, 255, 230])
_MORPH = np.ones((7, 7), np.uint8)


def chromatic_normalize(img_rgb):
    result = img_rgb.astype(np.float32)
    avg = result.reshape(-1, 3).mean(axis=0)
    scale = avg.mean() / (avg + 1e-6)
    return np.clip(result * scale, 0, 255).astype(np.uint8)


def pseudo_leaf_mask(img_rgb):
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, _HSV_GREEN_LOW, _HSV_GREEN_HIGH)
    lesion = cv2.inRange(hsv, _HSV_LESION_LOW, _HSV_LESION_HIGH)
    leaf = cv2.bitwise_or(green, lesion)
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_CLOSE, _MORPH)
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_OPEN, _MORPH)
    return (leaf > 0).astype(np.uint8)


def coco_train_test_split(coco, test_fraction=TEST_FRACTION, seed=42):
    ids = sorted(coco.getImgIds())
    np.random.RandomState(seed).shuffle(ids)
    n_test = round(len(ids) * test_fraction)
    return ids[n_test:], ids[:n_test]


def coco_leaf_mask(coco, img_id, size):
    info = coco.loadImgs(img_id)[0]
    leaf = np.zeros((info['height'], info['width']), np.uint8)
    for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
        leaf = np.maximum(leaf, coco.annToMask(ann))
    return info, cv2.resize(leaf, size, interpolation=cv2.INTER_NEAREST)


print('preprocesamiento y helpers COCO definidos OK')

In [ ]:
_SEG_AUG = A.Compose([
    A.Rotate(limit=40, border_mode=0, p=0.6),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.4),
    A.RandomShadow(p=0.2),
    A.Affine(translate_percent=0.08, scale=(0.85, 1.15), rotate=0, p=0.4),
    A.GaussNoise(std_range=(0.01, 0.06), p=0.2),
])


def _read_rgb(fp, size):
    for _ in range(3):
        try:
            return np.array(Image.open(fp).convert('RGB').resize(size))
        except (OSError, IOError):
            time.sleep(0.3)
    return None


class LeafSegSequence(Sequence):
    def __init__(self, coco, coco_ids, masks_dir, pseudo_paths, img_size, batch_size, augment, shuffle):
        self.coco = coco
        self.coco_ids = list(coco_ids)
        self.masks_dir = Path(masks_dir)
        self.pseudo_paths = list(pseudo_paths)
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self._rebuild()

    def _rebuild(self):
        pseudo = list(self.pseudo_paths)
        if self.shuffle and pseudo:
            np.random.shuffle(pseudo)
        pseudo = pseudo[:PSEUDO_PER_EPOCH]
        self.items = [('coco', i) for i in self.coco_ids] + [('pseudo', p) for p in pseudo]
        if self.shuffle:
            np.random.shuffle(self.items)

    def __len__(self):
        return max(1, len(self.items) // self.batch_size)

    def _load(self, kind, ref):
        if kind == 'coco':
            info, leaf = coco_leaf_mask(self.coco, ref, self.img_size)
            img = _read_rgb(self.masks_dir / info['file_name'], self.img_size)
        else:
            img = _read_rgb(ref, self.img_size)
            leaf = pseudo_leaf_mask(img) if img is not None else None
        if img is None or leaf is None:
            return None, None
        return chromatic_normalize(img), leaf.astype(np.uint8)

    def __getitem__(self, idx):
        batch = self.items[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, masks = [], []
        for kind, ref in batch:
            img, leaf = self._load(kind, ref)
            if img is None:
                continue
            if self.augment:
                aug = _SEG_AUG(image=img, mask=leaf)
                img, leaf = aug['image'], aug['mask']
            imgs.append(img.astype(np.float32) / 255.0)
            masks.append(leaf[..., np.newaxis].astype(np.float32))
        if not imgs:
            return np.zeros((1, *self.img_size, 3), np.float32), np.zeros((1, *self.img_size, 1), np.float32)
        return np.stack(imgs), np.stack(masks)

    def on_epoch_end(self):
        if self.shuffle:
            self._rebuild()


from pycocotools.coco import COCO

coco = COCO(str(COCO_ANN))
train_ids, test_ids = coco_train_test_split(coco)
_tr = list(train_ids)
np.random.RandomState(123).shuffle(_tr)
n_val = max(1, round(len(_tr) * VAL_FRACTION))
val_ids, fit_ids = _tr[:n_val], _tr[n_val:]

pseudo_paths = []
if CLS_DIR.exists():
    for cls_dir in CLS_DIR.iterdir():
        if cls_dir.is_dir():
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
                pseudo_paths.extend(str(p) for p in cls_dir.glob(ext))

train_gen = LeafSegSequence(coco, fit_ids, MASKS_DIR, pseudo_paths, IMG_SIZE, BATCH, augment=True, shuffle=True)
val_gen = LeafSegSequence(coco, val_ids, MASKS_DIR, [], IMG_SIZE, BATCH, augment=False, shuffle=False)
print(f'COCO: fit={len(fit_ids)} val={len(val_ids)} test held-out={len(test_ids)} | pseudo-hoja disponibles={len(pseudo_paths)}')

In [ ]:
def _decoder_block(x, skip, filters, l2):
    x = tf.keras.layers.UpSampling2D(2)(x)
    if skip is not None:
        x = tf.keras.layers.Concatenate()([x, skip])
    reg = tf.keras.regularizers.l2(l2)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    return tf.keras.layers.Activation('relu')(x)


def build_mseg(l2=L2_DEFAULT, dropout_rate=0.3):
    backbone = tf.keras.applications.ResNet50(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
    backbone.trainable = False
    s1 = backbone.get_layer('conv1_relu').output
    s2 = backbone.get_layer('conv2_block3_out').output
    s3 = backbone.get_layer('conv3_block4_out').output
    s4 = backbone.get_layer('conv4_block6_out').output
    x = _decoder_block(backbone.output, s4, 512, l2)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = _decoder_block(x, s3, 256, l2)
    x = _decoder_block(x, s2, 128, l2)
    x = _decoder_block(x, s1, 64, l2)
    x = tf.keras.layers.UpSampling2D(2)(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Conv2D(NUM_CLASSES, 1, activation='softmax')(x)
    return tf.keras.Model(backbone.input, x, name='M_seg_resnet50')


mseg = build_mseg()
mseg.summary(line_length=100)

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_oh = tf.one_hot(tf.squeeze(tf.cast(y_true, tf.int32), -1), NUM_CLASSES)
    total = tf.constant(0.0)
    for c in range(NUM_CLASSES):
        p = y_pred[..., c]
        t = tf.cast(y_true_oh[..., c], tf.float32)
        inter = tf.reduce_sum(p * t)
        union = tf.reduce_sum(p) + tf.reduce_sum(t)
        total += 1.0 - (2 * inter + smooth) / (union + smooth)
    return total / tf.cast(NUM_CLASSES, tf.float32)


def seg_loss(y_true, y_pred):
    ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(ce) + dice_loss(y_true, y_pred)


class DiceCoef(tf.keras.metrics.Metric):
    def __init__(self, class_id=1, name='dice_hoja', **kw):
        super().__init__(name=name, **kw)
        self.class_id = class_id
        self.inter = self.add_weight(name='inter', shape=(), initializer='zeros')
        self.union = self.add_weight(name='union', shape=(), initializer='zeros')

    def get_config(self):
        base = super().get_config(); base['class_id'] = self.class_id; return base

    def update_state(self, y_true, y_pred, sample_weight=None):
        pred_c = tf.cast(tf.argmax(y_pred, -1) == self.class_id, tf.float32)
        true_c = tf.cast(tf.squeeze(y_true, -1) == self.class_id, tf.float32)
        self.inter.assign_add(tf.reduce_sum(pred_c * true_c))
        self.union.assign_add(tf.reduce_sum(pred_c) + tf.reduce_sum(true_c))

    def result(self):
        return (2 * self.inter + 1e-6) / (self.union + 1e-6)

    def reset_state(self):
        self.inter.assign(0.0); self.union.assign(0.0)


class LeafRecall(tf.keras.metrics.Metric):
    def __init__(self, name='leaf_recall', **kw):
        super().__init__(name=name, **kw)
        self.tp = self.add_weight(name='tp', shape=(), initializer='zeros')
        self.pos = self.add_weight(name='pos', shape=(), initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        pred_c = tf.cast(tf.argmax(y_pred, -1) == 1, tf.float32)
        true_c = tf.cast(tf.squeeze(y_true, -1) == 1, tf.float32)
        self.tp.assign_add(tf.reduce_sum(pred_c * true_c))
        self.pos.assign_add(tf.reduce_sum(true_c))

    def result(self):
        return (self.tp + 1e-6) / (self.pos + 1e-6)

    def reset_state(self):
        self.tp.assign(0.0); self.pos.assign(0.0)


def compile_mseg(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=seg_loss,
                  metrics=[DiceCoef(1, 'dice_hoja'), LeafRecall()])


print('losses y metricas definidas OK')

In [ ]:
OUT.mkdir(exist_ok=True)
for layer in mseg.layers:
    layer.trainable = not layer.name.startswith('conv') or 'block' not in layer.name
for layer in mseg.layers:
    if any(layer.name.startswith(p) for p in ('up_sampling', 'conv2d', 'concatenate', 'dropout', 'batch_normalization', 'activation')):
        layer.trainable = True

compile_mseg(mseg, LR_P1)
cbs1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_leaf_recall', mode='max', patience=8, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(OUT / 'model_seg_p1_best.keras'), monitor='val_leaf_recall', mode='max', save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
]
print('=== FASE 1: decoder (encoder congelado) ===')
h1 = mseg.fit(train_gen, validation_data=val_gen, epochs=EPOCHS_P1, callbacks=cbs1, verbose=1)

In [ ]:
for layer in mseg.layers:
    if any(s in layer.name for s in ('conv4_block', 'conv5_block')) and not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

steps = max(1, len(train_gen)) * EPOCHS_P2
lr_sched = tf.keras.optimizers.schedules.CosineDecay(LR_P2, steps, alpha=0.05)
compile_mseg(mseg, lr_sched)
initial_epoch = len(h1.history['loss'])
cbs2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_leaf_recall', mode='max', patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(OUT / 'model_seg_best.keras'), monitor='val_leaf_recall', mode='max', save_best_only=True, verbose=1),
]
print('=== FASE 2: fine-tune ResNet50 conv4+conv5 ===')
h2 = mseg.fit(train_gen, validation_data=val_gen, epochs=initial_epoch + EPOCHS_P2, initial_epoch=initial_epoch, callbacks=cbs2, verbose=1)
mseg.save(OUT / 'model_seg.keras')
print('Modelo guardado en', OUT / 'model_seg.keras')

In [ ]:
loss_all = h1.history['loss'] + h2.history['loss']
vloss = h1.history['val_loss'] + h2.history['val_loss']
rec = h1.history['leaf_recall'] + h2.history['leaf_recall']
vrec = h1.history['val_leaf_recall'] + h2.history['val_leaf_recall']
ep = range(1, len(loss_all) + 1)
div = len(h1.history['loss'])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(ep, loss_all, 'b-', label='train'); axes[0].plot(ep, vloss, 'r-', label='val')
axes[0].axvline(div, color='gray', ls='--'); axes[0].set_title('M_seg - Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(ep, rec, 'b-', label='train'); axes[1].plot(ep, vrec, 'r-', label='val')
axes[1].axvline(div, color='gray', ls='--'); axes[1].set_title('M_seg - Recall hoja'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.savefig(OUT / 'M_seg_curves.png', dpi=120); plt.show()

gap = vloss[-1] - loss_all[-1]
print(f'train_loss={loss_all[-1]:.4f} val_loss={vloss[-1]:.4f} gap={gap:.4f}')
print('  ALTA VARIANZA (subir dropout/aug/L2)' if gap > 0.15 else ('  ALTO BIAS (mas capacidad/epocas)' if loss_all[-1] > 0.4 else '  Bias/varianza balanceados'))

In [ ]:
def evaluate_leaf(model, coco, ids, masks_dir, size):
    recalls, dices, ious = [], [], []
    for img_id in ids:
        info, gt = coco_leaf_mask(coco, img_id, size)
        raw = _read_rgb(Path(masks_dir) / info['file_name'], size)
        if raw is None:
            continue
        norm = chromatic_normalize(raw)
        prob = model.predict((norm.astype(np.float32) / 255.0)[np.newaxis], verbose=0)[0]
        pred = (np.argmax(prob, -1) == 1).astype(np.uint8)
        g = (gt == 1).astype(np.uint8)
        inter = int(np.sum((pred == 1) & (g == 1)))
        psum, gsum = int(pred.sum()), int(g.sum())
        recalls.append((inter + 1e-6) / (gsum + 1e-6))
        dices.append((2 * inter + 1e-6) / (psum + gsum + 1e-6))
        ious.append((inter + 1e-6) / (psum + gsum - inter + 1e-6))
    return float(np.mean(recalls)), float(np.mean(dices)), float(np.mean(ious))


rec_t, dice_t, iou_t = evaluate_leaf(mseg, coco, test_ids, MASKS_DIR, IMG_SIZE)
print(f'=== TEST vs experto COCO (n={len(test_ids)}, held-out 25% seed=42) ===')
print(f'Recall hoja (principal): {rec_t:.3f}')
print(f'Dice hoja  (secundario): {dice_t:.3f}')
print(f'IoU  hoja  (secundario): {iou_t:.3f}')
json.dump({'leaf_recall': round(rec_t, 4), 'dice': round(dice_t, 4), 'iou': round(iou_t, 4), 'n_test': len(test_ids)},
          open(OUT / 'mseg_test_metrics.json', 'w'), indent=2)

In [ ]:
_COLORS = np.array([[20, 20, 20], [34, 197, 94]], dtype=np.uint8)
_show = test_ids[:5]
fig, axes = plt.subplots(len(_show), 3, figsize=(11, 3 * len(_show)))
for r, img_id in enumerate(_show):
    info, gt = coco_leaf_mask(coco, img_id, IMG_SIZE)
    raw = _read_rgb(Path(MASKS_DIR) / info['file_name'], IMG_SIZE)
    norm = chromatic_normalize(raw)
    pred = np.argmax(mseg.predict((norm.astype(np.float32) / 255.0)[np.newaxis], verbose=0)[0], -1)
    axes[r, 0].imshow(raw); axes[r, 0].set_title('Original'); axes[r, 0].axis('off')
    axes[r, 1].imshow(_COLORS[(gt == 1).astype(int)]); axes[r, 1].set_title('Mascara experta'); axes[r, 1].axis('off')
    axes[r, 2].imshow(_COLORS[pred]); axes[r, 2].set_title('Pred M_seg'); axes[r, 2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(mseg)
(OUT / 'model_seg.tflite').write_bytes(converter.convert())
print('model_seg.tflite (float32):', round((OUT / 'model_seg.tflite').stat().st_size / 1e6, 2), 'MB')


def _rep():
    for i in range(min(60, len(val_gen))):
        x, _ = val_gen[i]
        for img in x:
            yield [img[np.newaxis].astype(np.float32)]


c8 = tf.lite.TFLiteConverter.from_keras_model(mseg)
c8.optimizations = [tf.lite.Optimize.DEFAULT]
c8.representative_dataset = _rep
c8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
c8.inference_input_type = tf.uint8
c8.inference_output_type = tf.uint8
(OUT / 'model_seg_int8.tflite').write_bytes(c8.convert())
print('model_seg_int8.tflite (int8):', round((OUT / 'model_seg_int8.tflite').stat().st_size / 1e6, 2), 'MB')